# K-means 클러스터링을 이용한 금융 사막화 지역 파악

### 사용 라이브러리

In [ ]:
import os
os.environ['OMP_NUM_THREADS'] = '8'
from tqdm import tqdm

# 수치 계산 라이브러리
import numpy as np
import pandas as pd
pd.options.mode.chained_assignment = None

# 시각화 라이브러리
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import seaborn as sns
import folium

# KMO 검정 및 Bartlett의 구형성 검정 수행을 위한 라이브러리
from factor_analyzer.factor_analyzer import calculate_kmo, calculate_bartlett_sphericity

# 주성분 분석을 위한 라이브러리
from sklearn.preprocessing import StandardScaler, QuantileTransformer
from sklearn.metrics import silhouette_score
from sklearn.model_selection import ParameterGrid
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

### 사용 함수

In [ ]:
def plot_distribution(df, cols=None, figsize=(10, 8), hue=None):
    '''데이터프레임을 받아 관심 컬럼들의 분포를 시각화합니다.'''
    if cols is None:
        cols = df.columns
    
    axes_rows = len(cols)//2 + len(cols)%2
    axes_cols = 2
    fig, ax = plt.subplots(axes_rows, axes_cols, figsize=figsize)

    for i, col in enumerate(cols):
        if col == 'estimatedAddressPop':
            # estimatedAddressPop을 사용하는 경우, kdeplot으로 시각화
            sns.kdeplot(df[col], ax=ax[i//2, i%2], hue=hue)
            ax[i//2, i%2].set_title(f'{col} Distribution')
        else:
            sns.histplot(df[col], ax=ax[i//2, i%2], hue=hue)
            ax[i//2, i%2].set_title(f'{col} Distribution')
        
    plt.tight_layout()
    plt.show()

### 클러스터링 방법 목차:
1. 사용 컬럼 
    - 1km 내 은행 개수 (numBanks1km)
    - 1~3km 내 은행 개수 편차 (numBanksStdDev)
    - 법정동 별 고령층 인구 비율 (areaPopPercOver70)
    - 가까운 은행까지 소요 시간 (minsToNearestBank)
1. IQR 사용한 이상치 제거
    * Quantile Transformation을 사용하기에 별도의 IQR을 이용한 이상치 제거는 하지 않음

2. 피쳐 스케일링
    - 표준화 (Quantile Transformation) 진행 
    - *(군집화 과정에 분포의 모양은 큰 영향을 끼치지 않음)*

3. 변수 간 상관관계 확인
    - Heatmap 및 구형성 검정 사용

4. Elbow Method / 실루엣 점수로 최적 군집 개수 계산

5. PCA 주성분 분석

6. Clustering을 한 데이터를 PCA 차원으로 시각화

7. 군집별 피쳐 분포 시각화
    - 이 값을 이용한 금융 사막화 지역 군집 정의

## 0. 데이터 불러오기

In [ ]:
raw_df = pd.read_parquet('./data/jeonnam_final.parquet')

In [ ]:
# 결측치 제거
raw_df = raw_df.dropna()
raw_df = raw_df.drop_duplicates(subset=['latitude', 'longitude'], keep='first')

raw_df.columns

### a. 피쳐 생성
반경 1, 1.5, 2, 2.5, 3km 내의 은행 개수의 편차를 구하여 새로운 컬럼으로 만들어 주겠습니다.

In [ ]:
# 원본 데이터에 생성
raw_df['은행개수 표준편차'] = raw_df[['1km_radius', '1.5km_radius', '2km_radius', '2.5km_radius', '3km_radius']].std(axis=1)

### b. 샘플 추출
법정동코드, 은행코드, 그리고 1km, 5km, 10km내 은행개수가 같은 샘플들은 같은 지역에 위치해 있는 지번들이라 생각하고 그 중 하나씩만 사용하겠습니다.

In [ ]:
# 지역 구분용 컬럼 정의
group_cols = ['법정동코드', 'closeness', '1km_radius', '5km_radius', '10km_radius']

# 위 명시된 컬럼들을 기준으로 위/경도 평균값 계산 (차후에 행정동 구할 때 사용)
raw_df[['mean_longitude', 'mean_latitude']] = raw_df.groupby(group_cols)[['longitude', 'latitude']].transform('mean')

In [ ]:
df_dropped = raw_df.drop_duplicates(subset=group_cols, keep='first')

print(f'샘플 추출 전: {len(raw_df)}개')
print(f'샘플 추출 후: {len(df_dropped)}개')

### c. 관심있는 컬럼 골라내기
군집화 및 지도 표기에 사용될 컬럼들만 뽑아 컬럼명을 통일화된 형태로 바꿔주겠다.

In [ ]:
# 사용할 컬럼 추출
extract_cols = ['address', 'latitude', 'longitude', 'closeness', 'distance', 'total_mins',
                '1km_radius', '1.5km_radius', '2km_radius', '2.5km_radius', '3km_radius', '은행개수 표준편차',
                '지번별 추정 인구수', '읍면동리 70세이상 인구비율', '읍면동 총인구', '읍면동리 총인구']

# CamelCase 형식으로 새로운 컬럼명 정의
new_cols = ['address', 'latitude', 'longitude', 'nearestBankCode', 'distanceToNearestBank', 'minsToNearestBank',
            'numBanks1km', 'numBanks1.5km', 'numBanks2km', 'numBanks2.5km', 'numBanks3km', 'numBanksStdDev',
            'estimatedAddressPop', 'popPercOver70', 'areaTotalPop', 'areaRiTotalPop']

# 컬럼명 변경
df = df_dropped[extract_cols]
df.columns = new_cols
df.drop('address', axis=1).head()

### d. 컬럼별 분포도 시각화

In [ ]:
# 관심 있는 컬럼만 추출
interested_cols = ['numBanks1km', 'numBanksStdDev', 'minsToNearestBank', 'popPercOver70']

In [ ]:
# 분포도 시각화 (seaborn 사용)
sns.set_theme()

plot_distribution(df, interested_cols)

## 1. IQR을 이용한 이상치 제거

In [ ]:
# IQR 기반 이상치 제거 함수
def remove_outliers_iqr(df, columns, threshold=1.5):
    df_temp = df.copy()
    for column in columns:
        Q1 = df_temp[column].quantile(0.25)
        Q3 = df_temp[column].quantile(0.75)
        IQR = Q3 - Q1
        
        lower_bound = Q1 - threshold * IQR
        upper_bound = Q3 + threshold * IQR
        
        df_temp = df_temp[(df_temp[column] > lower_bound) & (df_temp[column] < upper_bound)]
        
    return df_temp

#### *Quantile Transformation을 사용하면서 이상치 제거 과정은 스킵*

## 2. 피쳐 스케일링
### Quantile Transformation

법정동 70세이상 인구비율 분포를 제외하곤 다 한쪽으로 치우친 형태를 가지고 있는 것을 볼 수 있다.

모든 컬럼에 표준화를 진행해 주어 스케일을 동일하게 만들어 주겠다

In [ ]:
# 이상치 제거 후 데이터프레임
df_temp = df[interested_cols]

# 데이터 스케일링 (Quantile Transformer)
scaler = QuantileTransformer(output_distribution='normal')
scaled = scaler.fit_transform(df_temp)
scaled_df = pd.DataFrame(scaled, columns=interested_cols, index=df_temp.index)

In [ ]:
# 표준화 후 분포도 시각화
plot_distribution(scaled_df, interested_cols)

## 3. 변수 간 상관관계 확인

### a. 수치 시각화 (Heatmap)

In [ ]:
# 상관관계 계산
correlation = df[interested_cols].corr()

# Heatmap 시각화
plt.figure(figsize=(10, 7))
sns.heatmap(correlation, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap')
plt.xticks(rotation=0)
plt.show()

### b. 구형성 검정

In [ ]:
# KMO 검정 수행
kmo_all, kmo_model = calculate_kmo(df_temp)
print(f'KMO Model: {kmo_model}')

# Bartlett의 구형성 검정 수행
chi_square_value, p_value = calculate_bartlett_sphericity(df_temp)
print(f'Bartlett’s Test chi-square: {chi_square_value}')
print(f'Bartlett’s Test p-value: {p_value}')

KMO 값이 0.726으로 양호한 수준이며, Bartlett의 구형성 검정에서 p-value가 0.0으로 유의미한 차이를 나타낸다.

따라서, 이 데이터는 주성분 분석(PCA) 또는 요인 분석을 수행하기에 적합한 데이터라는 결론을 내릴 수 있다.

## 4. 군집화 진행

### a. Elbow Method를 사용하여 최적의 K 계산
K-means 군집화를 진행해주기에 앞서 최적을 K개수를 구하기 위해 Elbow method를 사용해 주었다.

In [ ]:
# Elbow Method를 사용하여 최적의 K 찾기
figure = plt.figure(figsize=(6, 4))
wcss = []
for i in range(1, 11):
    kmeans = KMeans(n_clusters=i, init='k-means++', random_state=42, n_init='auto')
    # kmeans.fit(df_IQR[interested_cols])
    kmeans.fit(scaled_df)
    wcss.append(kmeans.inertia_)
    
sns.lineplot(x=range(1, 11), y=wcss, marker='o')
plt.title('Elbow Method for Optimal K')
plt.xlabel('Number of clusters (K)')
plt.ylabel('WCSS')
plt.show()

### b. Silhouette 점수를 사용한 최적의 K값 계산

In [ ]:
# 데이터를 모두 사용하지 않고 일부만 사용하여 최적의 클러스터 개수 찾기 (데이터가 많을 때 유용)
subsample_fraction = 0.05  # 5%의 데이터만 사용
subsample = scaled_df[interested_cols].sample(frac=subsample_fraction, random_state=24)

# 가능한 클러스터 개수
parameters = np.arange(2, 10)

# Parameter Grid 생성
parameter_grid = ParameterGrid({'n_clusters': parameters})
best_score = -1
kmeans_model = KMeans(random_state=42, n_init=15, init='k-means++', max_iter=1000)

silhouette_scores = []
# 실루엣 점수를 기반으로 최적의 클러스터 개수 찾기
for p in tqdm(parameter_grid):
    kmeans_model.set_params(**p)    # 현 parameter로 모델 설정
    kmeans_model.fit(subsample)          # 데이터 학습
    ss = silhouette_score(subsample, kmeans_model.labels_)   # 실루엣 점수 계산
    silhouette_scores += [ss]       # 실루엣 점수 저장
    print('Parameter:', p, 'Score', ss)
    # 최고 점수 업데이트
    if ss > best_score:
        best_score = ss
        best_grid = p

In [ ]:
# 실루엣 점수 시각화
plt.bar(range(len(silhouette_scores)), list(silhouette_scores), align='center', width=0.5)
plt.xticks(range(len(silhouette_scores)), list(parameters))
plt.title('Silhouette Score', fontweight='bold')
plt.xlabel('Number of Clusters')
plt.show()

Elbow Method와 실루엣 스코어 둘 다 cluster 개수를 4개로 설정하는 것이 가장 이상적이라고 나타낸다.

### c. 군집화 진행

In [ ]:
# 군집화 결과를 저장할 데이터프레임 복사
cluster_df = scaled_df.copy()

# 최적의 클러스터 개수로 KMeans 클러스터링 수행
kmeans = KMeans(n_clusters=4, n_init='auto', random_state=42, max_iter=1000)
kmeans.fit(scaled_df)

# cluster 라벨 저장
cluster_labels = kmeans.labels_
cluster_df['cluster'] = cluster_labels

# 각 군집의 중심점 확인
cluster_centers = scaler.inverse_transform(kmeans.cluster_centers_)

# 각 군집의 중심점 출력
cluster_centers_df = pd.DataFrame(cluster_centers, columns=scaled_df.columns)

# 각 군집별 데이터 개수 확인
cluster_counts = cluster_df['cluster'].value_counts()

# 각 군집별 데이터 개수 출력
print("Cluster Counts:")
print(cluster_counts)
cluster_centers_df.sort_values(by=['numBanks1km', 'numBanksStdDev'])

## 5. PCA 주성분 분석

In [ ]:
figure = plt.figure(figsize=(10, 7))

# PCA를 통해 주성분 분석 (더 많은 주성분을 포함)
pca_all = PCA(n_components=None, random_state=42)  # 모든 주성분을 포함하여 PCA 수행
pca_result = pca_all.fit_transform(scaled_df)

# 주성분의 고유값과 설명된 분산 비율
explained_variance = pca_all.explained_variance_
explained_variance_ratio = pca_all.explained_variance_ratio_
cumulative_explained_variance_ratio = pca_all.explained_variance_ratio_.cumsum()

print('Explained Variance:', explained_variance)
print('Explained Variance Ratio:', explained_variance_ratio)
print('Cumulative Explained Variance Ratio:', cumulative_explained_variance_ratio)

# 누적 설명된 분산 비율 시각화
plt.plot(range(1, len(cumulative_explained_variance_ratio) + 1), cumulative_explained_variance_ratio, marker='o')
plt.xlabel('Number of Principal Components')
plt.ylabel('Cumulative Explained Variance Ratio')
plt.title('Cumulative Explained Variance Ratio by Principal Components')
plt.grid()
plt.show()

1. 주성분의 설명력:

- 첫 두 주성분 (PC1 + PC2)이 데이터의 약 92.84%의 분산을 설명
- 세 번째 주성분 (PC3)을 추가하면 설명된 분산 비율이 약 96.87%로 증가

2. 적합성:

    누적 설명된 분산 비율이 70% 이상이 되면 데이터의 특성을 잘 설명할 수 있다고 보기 때문에
PC1, PC2, 이 두 주성분만으로도 데이터의 상당 부분을 설명할 수 있다.

3. 주성분 개수 선택:

    두 개의 주성분 (PC1, PC2)만으로도 충분히 설명력이 높기 때문에, 이 두 주성분을 사용하는 것이 합리적일 수 있다.

In [ ]:
#PCA 로딩 값 계산
loadings = pd.DataFrame(pca_all.components_.T, columns=[f'PC{i+1}' for i in range(len(pca_all.explained_variance_))], 
                        index=scaled_df.columns)
loadings

## 6. PCA를 이용한 군집 시각화
관련 설명 내용: https://medium.com/swlh/k-means-clustering-on-high-dimensional-data-d2151e1a4240 

In [ ]:
# PCA를 통해 주성분 분석 (첫 세 주성분만 사용)
pca = PCA(n_components=3, random_state=88)
pca_result = pca.fit_transform(scaled_df)

# 주성분의 고유값과 설명된 분산 비율
dataset_pca = pd.DataFrame(abs(pca.components_), columns=scaled_df.columns, index=['PC_1', 'PC_2', 'PC_3'])
dataset_pca

In [ ]:
# 클러스터 중심을 PCA 공간으로 변환
pca_cluster_centers = pca.transform(scaler.transform(cluster_centers_df))

# PCA 결과 시각화 (3D 시각화)
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# 데이터 점 플롯
sc = ax.scatter(pca_result[:, 0], pca_result[:, 1], pca_result[:, 2], 
                c=cluster_df['cluster'], cmap='viridis', label='Data points', s=10, alpha=0.05)

# 클러스터 중심 레이블 추가
for i, center in enumerate(pca_cluster_centers):
    ax.text(center[0]+.5, center[1]+.5, center[2], f'Cluster {i}', color='black', fontsize=12, weight='bold')

ax.set_xlabel('Principal Component 1')
ax.set_ylabel('Principal Component 2')
ax.set_zlabel('Principal Component 3')
plt.title('PCA (3 components)')

plt.show()

---
---

## 7. 군집별 피쳐 분포 시각화

In [ ]:
def plot_cluster_dist_comparison(df, cols=None, data_scaled=True, figsize=(10, 7)):
    '''클러스터 별로 데이터를 스케일링 전으로 복원 후 비교용 히스토그램을 시각화 하는 함수'''
    if cols is None:
        cols = df.columns
    
    axes_rows = len(cols)//2 + len(cols)%2
    axes_cols = 2
    fig, ax = plt.subplots(axes_rows, axes_cols, figsize=figsize)
    
    color_list = ['purple', 'blue', 'green', 'yellow']

    # 클러스터 별로 데이터 복원
    for n in df['cluster'].unique():
        if data_scaled == True:
            # 스케일링 전으로 복원
            cluster_data = df[df['cluster'] == n]
            cluster_data = scaler.inverse_transform(cluster_data.drop('cluster', axis=1))
            cluster_data_df = pd.DataFrame(cluster_data, columns=interested_cols)
        else:
            # 데이터 그대로 사용
            cluster_data_df = df[df['cluster'] == n]
        
        # 각 클러스터별 히스토그램 시각화
        for i, col in enumerate(cols):
            if col in ['numBanks1km', 'numBanksStdDev']:
                bins = cluster_data_df[col].nunique()
                sns.histplot(cluster_data_df[col], ax=ax[i//2, i%2],
                             color=color_list[n], label='Cluster '+str(n), 
                             alpha=0.5, bins=bins, stat='density')
                ax[i//2, i%2].set_ylim(0, 1.2)
            else:
                bins = int((cluster_data_df[col].max() - cluster_data_df[col].min()))
                sns.kdeplot(cluster_data_df[col], ax=ax[i//2, i%2],
                            color=color_list[n], label='Cluster '+str(n))
            
            ax[i//2, i%2].legend()
            
            if col == 'minsToNearestBank':
                ax[i//2, i%2].set_xlim(-20, 200)
            elif col == 'popPercOver70':
                ax[i//2, i%2].set_xlim(-10, 80)
            else:
                pass
        
    plt.tight_layout()
    plt.show()

In [ ]:
plot_cluster_dist_comparison(cluster_df, cols=interested_cols, figsize=(10, 9))

---

### 클러스터 구분을 이전 데이터에 다시 적용

In [ ]:
# 인덱스를 기준으로 클러스터 라벨을 raw_df_copy에 추가
raw_df_copy = raw_df.copy()
raw_df_copy['cluster'] = cluster_df['cluster']

# 클러스터 라벨이 있는 데이터만 사용
raw_df_copy.dropna(subset=['cluster'], inplace=True)
raw_df_copy['cluster'] = raw_df_copy['cluster'].astype('int')  # 클러스터 라벨을 정수형으로 변환

In [ ]:
raw_df_copy.columns

In [ ]:
# 분류된 데이터를 저장
raw_df_copy.to_parquet('./data/jeonnam_clustered.parquet')

In [ ]:
# 금융 사막화 지역(클러스터 0)에 속한 데이터만 추출하여 저장
cluster_0_data = raw_df_copy[raw_df_copy['cluster'] == 0]
cluster_0_data.to_parquet(f'./data/banking_desert_samples.parquet')

---
## 8. 지도에서 군집 시각화

In [ ]:
# 지도 객체 생성
f = folium.Figure(width=800, height=600)
m = folium.Map(location=[raw_df_copy['mean_latitude'].mean(), raw_df_copy['mean_longitude'].mean()], tiles='cartodbpositron',
               zoom_start=9, min_zoom=9).add_to(f)

# 색깔 스키마 정의
color_schemes = {
    'kmeans': {
        'colors': {0: 'purple', 1: 'blue', 2: 'green', 3: 'yellow'},
        'opacities': {0: 0.6, 1: 0.6, 2: 0.6, 3: 0.6}
    },
    'banking_desert': {
        'colors': {0: 'red', 1: 'green', 2: 'orange', 3: 'green'},
        'opacities': {0: 0.6, 1: 0.6, 2: 0.6, 3: 0.6}
    },
    '0_only': {
        'colors': {0: 'red'},
        'opacities': {0: 0.6, 1: 0.1, 2: 0.1, 3: 0.1}
    }
}

# 색상 스키마별 레이어를 추가
for scheme_name, scheme in color_schemes.items():
    colors = scheme['colors']
    opacities = scheme['opacities']
    feature_group = folium.FeatureGroup(name=scheme_name)
    
    for index, row in raw_df_copy.iterrows():
        cluster = row['cluster']
        color = colors.get(cluster, 'grey')  # 색상이 명시되지 않으면 회색으로 표시
        opacity = opacities.get(cluster, 0.8)  # 투명도가 명시되지 않으면 0.8로 표시
        folium.CircleMarker(location=[row['mean_latitude'], row['mean_longitude']], 
                            radius=2, stroke=False, color=color, fill=True, 
                            fill_opacity=opacity).add_to(feature_group)
    
    feature_group.add_to(m)

# 레이어 컨트롤 추가
folium.LayerControl().add_to(m)

# 지도 저장 및 시각화
m

---
---